# Breast Cancer Exploratory Data Analysis (EDA)

## Objective

The objective of this notebook is to explore and understand the Breast Cancer Wisconsin Diagnostic Dataset before building machine learning models.

The analysis focuses on:

- Understanding the structure of the dataset
- Cleaning the data
- Exploring feature distributions
- Identifying relationships between variables
- Detecting potential outliers
- Drawing insights that can improve model performance

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

# Load the Dataset

The first step is to load the dataset and inspect its contents.

In [ ]:
data_path = Path("../data/data.csv")

df = pd.read_csv(data_path)

df.head()

# Dataset Overview

Before cleaning the data, it is important to understand its size, data types, and overall structure.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

In [ ]:
df.describe().T

## Initial Observations

From the initial inspection, we can observe:

- The dataset contains both identifier and diagnostic information.
- Most features are numerical measurements.
- The `id` column is not useful for prediction.
- The `Unnamed: 32` column appears to contain only missing values and will be removed during preprocessing.

# Data Cleaning

Before performing any analysis, we need to clean the dataset.

The cleaning process involves:

- Removing unnecessary columns.
- Converting the target variable into numerical values.
- Verifying that the dataset is ready for analysis.

In [ ]:
# Remove unnecessary columns
df = df.drop(columns=["id", "Unnamed: 32"])

# Convert diagnosis to numerical values
df["diagnosis"] = df["diagnosis"].map({
    "M": 1,
    "B": 0
})

df.head()

# Check for Missing Values

Missing values can affect both data analysis and machine learning models.

Let's verify whether any missing values remain after cleaning.

In [ ]:
df.isnull().sum()

## Observation

After removing the unnecessary column, the dataset contains no missing values.

This means no additional imputation or data filling is required before analysis.

# Class Distribution

Before training a classification model, it is important to examine the distribution of the target variable.

A balanced dataset generally helps machine learning models perform better.

In [ ]:
df["diagnosis"].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x="diagnosis",
    data=df
)

plt.xticks(
    [0,1],
    ["Benign","Malignant"]
)

plt.title("Distribution of Breast Cancer Diagnosis")

plt.xlabel("Diagnosis")
plt.ylabel("Number of Samples")

plt.show()

## Insight

The dataset contains more benign tumors than malignant tumors.

Although the classes are not perfectly balanced, the imbalance is relatively small and should not significantly affect model training.

# Summary Statistics

Summary statistics provide an overview of the numerical features, including:

- Mean
- Standard deviation
- Minimum
- Maximum
- Quartiles

These statistics help identify unusual values and understand the scale of each feature.

In [ ]:
df.describe().T

## Observation

The numerical features have very different scales.

For example, variables such as **area_mean** have much larger values than variables such as **smoothness_mean**.

This explains why feature scaling is required for algorithms like Logistic Regression, K-Nearest Neighbors, and Support Vector Machines.

# Feature Distributions

Understanding how each feature is distributed helps us identify:

- Skewed variables
- Normal distributions
- Potential outliers
- Differences in feature scales

Histograms provide a quick overview of the distribution of every numerical feature.

In [ ]:
df.hist(
    figsize=(20,18),
    bins=20,
    edgecolor="black"
)

plt.tight_layout()

plt.show()

## Observation

The histograms show that several variables are not normally distributed.

Some features are right-skewed, while others appear approximately bell-shaped.

This information is useful when selecting machine learning algorithms and considering feature transformations.

# Correlation Analysis

Correlation measures the strength of the relationship between two variables.

Highly correlated variables often carry similar information.

A correlation heatmap allows us to quickly identify these relationships.

In [ ]:
plt.figure(figsize=(18,15))

sns.heatmap(
    df.corr(),
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap")

plt.show()

## Insight

Several features exhibit strong positive correlations.

For example, measurements related to tumor size, such as radius, perimeter, and area, tend to increase together.

This suggests that some variables may contain overlapping information.

# Features Most Correlated with Diagnosis

Rather than examining every feature individually, it is useful to identify which variables are most strongly associated with the diagnosis.

These features are likely to be the most informative for classification.

In [ ]:
correlation = (
    df.corr()["diagnosis"]
    .sort_values(ascending=False)
)

correlation

In [ ]:
correlation

## Observation

The strongest relationships with the diagnosis are observed in features related to tumor size and shape.

These variables are expected to contribute significantly to the predictive performance of machine learning models.

# Pair Plot

Pair plots visualize relationships between multiple variables simultaneously.

Since plotting all features would produce an extremely large figure, a subset of the most informative variables is selected.

In [ ]:
selected_features = [

    "radius_mean",

    "texture_mean",

    "perimeter_mean",

    "area_mean",

    "diagnosis"

]

sns.pairplot(
    df[selected_features],
    hue="diagnosis"
)

plt.show()

## Insight

The pair plots reveal a noticeable separation between benign and malignant tumors.

This indicates that the selected features possess strong discriminative power and are likely to support accurate classification.

# Boxplots

Boxplots are used to identify the spread of the data and detect potential outliers.

Outliers are observations that differ significantly from the rest of the dataset.

In [ ]:
plt.figure(figsize=(20,8))

sns.boxplot(data=df)

plt.xticks(rotation=90)

plt.title("Boxplots of All Features")

plt.show()

## Observation

Several numerical features contain outliers.

Because this is a medical dataset, these values may represent genuine biological variations rather than data entry errors.

For this reason, they are retained during model development.

# Relationship Between Radius and Area

Scatter plots help visualize the relationship between two continuous variables.

Here, we investigate the relationship between tumor radius and tumor area while considering the diagnosis.

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=df,
    x="radius_mean",
    y="area_mean",
    hue="diagnosis"
)

plt.title("Radius Mean vs Area Mean")

plt.show()

## Insight

Tumors with larger radius measurements generally have larger areas.

Malignant tumors tend to cluster in the region representing larger radius and area values, suggesting that tumor size is an important indicator of malignancy.

# Feature Importance

Although feature importance is typically part of the modeling stage, it can also provide valuable insights during exploratory analysis.

In this section, we use a Random Forest model to estimate which features contribute the most to predicting whether a tumor is benign or malignant.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Separate features and target
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

# Train a Random Forest model
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

# Create a DataFrame of feature importances
feature_importance = (
    pd.DataFrame({
        "Feature": X.columns,
        "Importance": rf.feature_importances_
    })
    .sort_values(by="Importance", ascending=False)
)

feature_importance.head(10)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance.head(10),
    x="Importance",
    y="Feature"
)

plt.title("Top 10 Most Important Features")

plt.xlabel("Importance Score")
plt.ylabel("Feature")

plt.show()

## Observation

The Random Forest model identifies features related to tumor size, perimeter, radius, and concavity as the most influential predictors.

These findings are consistent with the earlier correlation analysis, reinforcing their importance in distinguishing between benign and malignant tumors.

# Save the Cleaned Dataset

To ensure reproducibility and avoid repeating preprocessing steps, the cleaned dataset is saved for future use.

In [ ]:
from pathlib import Path

output_path = Path("../data/clean_data.csv")

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")

# Key Insights

The exploratory data analysis revealed several important findings:

- The dataset contains no missing values after removing unnecessary columns.
- Benign tumors are more common than malignant tumors, although the class imbalance is relatively small.
- Several features exhibit strong positive correlations, particularly those related to tumor size.
- Variables such as radius, perimeter, area, and concavity show the strongest relationships with the diagnosis.
- Some numerical features contain outliers, but these are likely to represent genuine medical observations rather than errors.
- The feature importance analysis confirms that measurements describing tumor size and shape are the most influential predictors.

Overall, the dataset is clean, well-structured, and suitable for building reliable machine learning classification models.

# Conclusion

This exploratory data analysis provided a comprehensive understanding of the Breast Cancer Wisconsin Diagnostic Dataset.

The analysis demonstrated that the dataset is suitable for classification tasks and highlighted the variables most strongly associated with breast cancer diagnosis.

The insights obtained during this stage support the machine learning workflow implemented in the project and provide a strong foundation for developing accurate predictive models.